# Oracle AIDP — MLOps end-to-end demo
### Predicting next-day direction for well-known large-cap stocks

This notebook is a **live demo script** for AIDP's MLOps capabilities. It runs entirely inside an
AIDP Workbench notebook — MLflow is already wired to the AIDP MLOps service, so there is no
tracking URI, token or endpoint to configure.

**What gets demonstrated, in order**

| # | Capability | What the audience sees |
|---|------------|------------------------|
| 1 | Synthetic market data | Familiar tickers — ORCL, AAPL, MSFT, NVDA, AMZN, GOOGL, TSLA, JPM |
| 2 | **Experiment tracking** | One experiment, three runs, params + metrics + tags + dataset lineage |
| 3 | **Model registry** | Three versions of *one* registered model, `catalog.schema.name` governed |
| 4 | **Input/output contract** | An MLflow signature + input example that AIDP reads and surfaces |
| 5 | **Version comparison** | A leaderboard, a champion, an aliased challenger |
| 6 | **Serving** | Deploy from the UI, then call `/predict` with a JSON request and read the response |

**Business framing for the room:** *"Every morning, before the open, we score eight names and tell the
desk which ones are likely to close higher tomorrow — with a probability and a confidence band. Three
data scientists tried three approaches. AIDP tracked all of them, kept the API contract identical, and
let us promote the winner without changing a single line of client code."*

> The market data is **synthetic and generated in this notebook** — no data source or entitlement is
> needed to run the demo, and nothing here is investment advice.

## 0 — Environment check and demo configuration

In AIDP Workbench, `mlflow` is pre-configured against the workspace MLOps service. The registered
model name **must** be a three-part `catalog.schema.name`, and that catalog + schema must already
exist in the workspace — AIDP validates this on registration.

This cell also resolves the **exact `==` pins** that every model version will be logged with. The
serving container rebuilds the environment from the model's `requirements.txt`, and a pickled
scikit-learn pipeline plus a cloudpickle-serialised `PythonModel` can only be loaded back by the
same versions that wrote them — so this kernel is the source of truth for the deployment
environment. We pin the six packages that actually matter at serving time instead of letting
MLflow freeze the whole workbench image.

In [3]:
import importlib.metadata
import json, sys, time, warnings
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# --- Managed-runtime pandas options: MUST run before any DataFrame is built ----------------
# AIDP Workbench turns on pandas' future string inference, which makes every string column a pandas
# StringDtype (Arrow-backed) instead of numpy object dtype. That breaks anything that hands the dtype
# to numpy, and MLflow's input-example writer trips over an Arrow-backed *columns* Index too. Turning
# both options off is what every AIDP customer-demo notebook does.
def configure_pandas_for_managed_runtime():
    """Force numpy-object strings in pandas. Returns a status string for the summary below."""
    states = []
    for option, value in (("future.infer_string", False), ("mode.string_storage", "python")):
        try:
            pd.set_option(option, value)
            states.append(f"{option}={pd.get_option(option)!r}")
        except Exception as exc:            # option absent on this pandas: nothing to turn off
            states.append(f"{option}=unavailable ({type(exc).__name__})")
    probe = pd.DataFrame({"s": ["AAPL"]})["s"].dtype
    states.append(f"string columns -> {probe!r}")
    if probe != object:
        states.append("WARNING: string columns are still not numpy object dtype")
    return "; ".join(states)


PANDAS_STRING_MODE = configure_pandas_for_managed_runtime()


def with_object_columns(frame):
    """Force the column *names* to numpy object dtype, as the AIDP reference notebooks do."""
    frame = pd.DataFrame(frame).copy()
    frame.columns = pd.Index([str(column) for column in frame.columns], dtype="object")
    return frame


# --- Demo configuration -------------------------------------------------------------------
CATALOG, SCHEMA = "default", "default"      # must already exist in the workspace
MODEL_NAME = f"{CATALOG}.{SCHEMA}.stock_direction_predictor_sklearn"
EXPERIMENT_NAME = "stock_direction_demo_sklearn"
DEMO_SEED = 3
# ------------------------------------------------------------------------------------------

# --- Pinned serving requirements ----------------------------------------------------------
# Logged as the model's requirements.txt, taken from THIS kernel so the rebuilt serving env
# matches the env that pickled the model. AIDP MLOps runs MLflow 3.15.x.
MLFLOW_SERIES = "3.12"
SERVING_LIBS = ["scikit-learn", "numpy", "scipy", "pandas", "cloudpickle"]


def resolve_pip_requirements(packages=SERVING_LIBS, mlflow_series=MLFLOW_SERIES):
    """Exact '==' pins for the running kernel: ['mlflow==3.15.0', 'scikit-learn==1.x.y', ...]."""
    if not mlflow.__version__.startswith(f"{mlflow_series}."):
        raise RuntimeError(f"this demo targets MLflow {mlflow_series}.x, "
                           f"kernel has {mlflow.__version__} - the logged pin must match "
                           f"the MLOps runtime or deployment will resolve a different MLflow")
    pins = [f"mlflow=={mlflow.__version__}"]      # keep 'mlflow' (not mlflow-skinny): serving needs it
    missing = []
    for dist in packages:
        try:
            pins.append(f"{dist}=={importlib.metadata.version(dist)}")
        except importlib.metadata.PackageNotFoundError:
            missing.append(dist)
    if missing:
        raise RuntimeError(f"cannot pin, not installed in this kernel: {missing}")
    return pins


PIP_REQUIREMENTS = resolve_pip_requirements()
# ------------------------------------------------------------------------------------------

client = MlflowClient()
print(f"python        : {sys.version.split()[0]}")
print(f"mlflow        : {mlflow.__version__}")
print(f"tracking uri  : {mlflow.get_tracking_uri()}")
print(f"registry uri  : {mlflow.get_registry_uri()}")
print(f"experiment    : {EXPERIMENT_NAME}")
print(f"model name    : {MODEL_NAME}")
print(f"pandas strings: {PANDAS_STRING_MODE}")
print("\nrequirements.txt that every version will be logged with:")
for pin in PIP_REQUIREMENTS:
    print(f"  {pin}")

python        : 3.11.13
mlflow        : 3.12.0
tracking uri  : https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/
registry uri  : https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/
experiment    : stock_direction_demo_sklearn
model name    : default.default.stock_direction_predictor_sklearn
pandas strings: future.infer_string=False; mode.string_storage="python"; string columns -> dtype("O")

requirements.txt that every version will 

## 1 — Synthetic market data for eight household-name stocks

Six years of daily closes and volumes for **ORCL, AAPL, MSFT, NVDA, AMZN, GOOGL, TSLA, JPM**.

The generator is not a random walk — it deliberately embeds a *learnable* edge, so the demo shows
real model improvement rather than three coin flips:

* **RSI mean-reversion** — stretched names snap back (any model can pick this up).
* **Volume confirming the trend** — for momentum names (NVDA, TSLA, AMZN) heavy volume in the
  direction of the trend continues; for the steadier names (ORCL, AAPL, MSFT, GOOGL, JPM) it fades.
  *A model only gets this if it can see `volume_ratio` **and** knows which ticker it is looking at.*
* **Volume exhaustion** — an unusually heavy session (>1.5× normal) flips the reversion edge.

That is the whole story of why v2 beats v1 and v3 beats v2.

In [4]:
TICKERS = {
    "ORCL":  dict(start=118.0, drift=0.20, vol=0.29, beta=1.00, style="reversion", sector="Enterprise Software"),
    "AAPL":  dict(start=190.0, drift=0.14, vol=0.26, beta=1.10, style="reversion", sector="Consumer Tech"),
    "MSFT":  dict(start=375.0, drift=0.16, vol=0.25, beta=1.05, style="reversion", sector="Enterprise Software"),
    "NVDA":  dict(start=520.0, drift=0.28, vol=0.46, beta=1.55, style="momentum",  sector="Semiconductors"),
    "AMZN":  dict(start=155.0, drift=0.13, vol=0.31, beta=1.25, style="momentum",  sector="E-Commerce"),
    "GOOGL": dict(start=142.0, drift=0.13, vol=0.27, beta=1.15, style="reversion", sector="Internet"),
    "TSLA":  dict(start=245.0, drift=0.06, vol=0.50, beta=1.65, style="momentum",  sector="Automotive"),
    "JPM":   dict(start=172.0, drift=0.10, vol=0.22, beta=0.85, style="reversion", sector="Banking"),
}
EDGE_STRENGTH = 1.45   # how much of tomorrow's move is predictable from today's features


def _rsi_14(closes):
    """Wilder-style RSI over the last 14 daily changes."""
    diffs = np.diff(closes[-15:])
    gain, loss = np.mean(np.clip(diffs, 0, None)), np.mean(np.clip(-diffs, 0, None))
    return 100.0 if loss == 0 else 100 - 100 / (1 + gain / loss)


def generate_market(seed=DEMO_SEED, n_days=1512):
    """~6 years of business-day closes and volumes with a learnable next-day edge."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(end=pd.Timestamp("2025-12-31"), periods=n_days)

    market = rng.normal(0.0003, 0.007, n_days)                 # common market factor
    market[rng.choice(n_days, 16, replace=False)] -= 0.025     # a handful of risk-off days

    regime, state = np.ones(n_days), 1.0                       # persistent calm / stressed regimes
    for t in range(n_days):
        if rng.random() < (0.02 if state == 1.0 else 0.06):
            state = 1.9 if state == 1.0 else 1.0
        regime[t] = state

    frames = []
    for ticker, cfg in TICKERS.items():
        daily_vol = cfg["vol"] / np.sqrt(252)
        idio = np.sqrt(max(daily_vol ** 2 - (cfg["beta"] * 0.007) ** 2, (0.3 * daily_vol) ** 2))
        mu = cfg["drift"] / 252 - 0.5 * daily_vol ** 2
        style_sign = 1.0 if cfg["style"] == "momentum" else -1.0
        volume = rng.lognormal(16.0, 0.38, n_days)
        closes = [cfg["start"]]

        for t in range(1, n_days):
            history, edge = np.array(closes), 0.0
            if t >= 21:
                sma5, sma20 = history[-5:].mean(), history[-20:].mean()
                rsi = _rsi_14(history)
                vol_ratio = volume[t - 1] / volume[t - 21:t].mean()
                rsi_dev = -(rsi - 50) / 15.0                       # stretched -> snap back
                vol_surprise = np.clip(vol_ratio - 1.0, -0.8, 2.0)
                trend_dir = np.sign(sma5 / sma20 - 1.0)
                heavy = 1.0 if vol_ratio > 1.5 else 0.0
                edge = float(np.clip(
                    0.50 * rsi_dev                                   # visible to every model
                    + 0.80 * vol_surprise * trend_dir * style_sign   # needs volume + ticker identity
                    - 0.75 * heavy * rsi_dev,                        # heavy volume flips the edge
                    -2.5, 2.5))
            ret = (mu + EDGE_STRENGTH * idio * edge
                   + cfg["beta"] * market[t] * regime[t]
                   + rng.normal(0, idio) * regime[t])
            closes.append(closes[-1] * np.exp(ret))

        closes = np.array(closes)
        traded = volume * (1 + 2 * np.abs(np.r_[0.0, np.diff(np.log(closes))]))
        frames.append(pd.DataFrame({"trade_date": dates, "ticker": ticker,
                                    "sector": cfg["sector"],
                                    "close": closes.round(2), "volume": traded.round(0)}))
    return pd.concat(frames, ignore_index=True)


prices = generate_market()

summary = (prices.groupby("ticker")
           .agg(first_close=("close", "first"), last_close=("close", "last"),
                sessions=("close", "size"))
           .assign(total_return=lambda d: (d.last_close / d.first_close - 1).mul(100).round(1)))
print(f"{len(prices):,} rows | {prices.trade_date.min():%Y-%m-%d} -> {prices.trade_date.max():%Y-%m-%d}\n")
print(summary.sort_values("total_return", ascending=False).to_string())

12,096 rows | 2020-03-17 -> 2025-12-31

        first_close  last_close  sessions  total_return
ticker                                                 
MSFT          375.0      863.98      1512         130.4
NVDA          520.0      844.94      1512          62.5
ORCL          118.0      185.87      1512          57.5
TSLA          245.0      355.45      1512          45.1
AMZN          155.0      221.39      1512          42.8
JPM           172.0      234.13      1512          36.1
AAPL          190.0      240.17      1512          26.4
GOOGL         142.0      138.64      1512          -2.4


## 2 — Features and the prediction target

Seven plain, explainable indicators any trader recognises. The label is simply
**"does this name close higher tomorrow?"**

The split is **chronological** (first 80% train, last 20% test) — never random for time series.

In [5]:
FEATURE_COLUMNS = ["ticker", "close", "sma_5", "sma_20", "rsi_14",
                   "volatility_20d", "volume_ratio", "momentum_5d"]


def build_features(df):
    frames = []
    for _, g in df.groupby("ticker", sort=False):
        g = g.sort_values("trade_date").copy()
        returns = g["close"].pct_change()
        delta = g["close"].diff()
        avg_gain = delta.clip(lower=0).rolling(14).mean()
        avg_loss = (-delta.clip(upper=0)).rolling(14).mean().replace(0, np.nan)

        g["sma_5"] = g["close"].rolling(5).mean()
        g["sma_20"] = g["close"].rolling(20).mean()
        g["rsi_14"] = 100 - 100 / (1 + avg_gain / avg_loss)
        g["volatility_20d"] = returns.rolling(20).std() * np.sqrt(252)
        g["volume_ratio"] = g["volume"] / g["volume"].rolling(20).mean()
        g["momentum_5d"] = g["close"] / g["close"].shift(5) - 1
        g["target_up"] = (g["close"].shift(-1) > g["close"]).astype(int)   # <- what we predict
        frames.append(g)

    out = pd.concat(frames).dropna().reset_index(drop=True)
    numeric = [c for c in FEATURE_COLUMNS if c != "ticker"]
    out[numeric] = out[numeric].astype("float64").round(6)
    return out


features = build_features(prices)
split_date = features["trade_date"].quantile(0.8)
train_df = features[features["trade_date"] <= split_date]
test_df = features[features["trade_date"] > split_date]

X_train, y_train = train_df[FEATURE_COLUMNS], train_df["target_up"]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df["target_up"]

print(f"train : {len(X_train):,} rows  (to {split_date:%Y-%m-%d})")
print(f"test  : {len(X_test):,} rows  (from {test_df.trade_date.min():%Y-%m-%d})")
BASELINE_ACCURACY = max(y_test.mean(), 1 - y_test.mean())   # always-guess-the-majority-class
print(f"UP days in test        : {y_test.mean():.1%}")
print(f"baseline accuracy      : {BASELINE_ACCURACY:.1%}  <- the number to beat\n")
print(X_train.head(4).to_string(index=False))

train : 9,552 rows  (to 2024-11-08)
test  : 2,384 rows  (from 2024-11-11)
UP days in test        : 49.6%
baseline accuracy      : 50.4%  <- the number to beat

ticker  close   sma_5   sma_20    rsi_14  volatility_20d  volume_ratio  momentum_5d
  ORCL 107.16 104.912 109.3440 44.507442        0.437101      0.807973     0.004688
  ORCL 109.55 106.038 109.2815 46.018614        0.391789      0.511783     0.054176
  ORCL 110.49 107.422 109.3540 54.252088        0.388476      0.622369     0.066815
  ORCL 108.63 108.222 109.1085 36.163378        0.362350      1.046599     0.038230


## 3 — The input/output contract

This is the part that makes the model *deployable* rather than just *trained*.

Every version is wrapped in an `mlflow.pyfunc.PythonModel` so the **API contract stays frozen**
while the algorithm underneath changes freely:

**Request — 8 fields per row**

| field | type | meaning |
|-------|------|---------|
| `ticker` | string | one of the eight symbols |
| `close` | double | latest close |
| `sma_5`, `sma_20` | double | 5- and 20-day simple moving averages |
| `rsi_14` | double | relative strength index, 0–100 |
| `volatility_20d` | double | annualised 20-day realised volatility |
| `volume_ratio` | double | today's volume ÷ 20-day average volume |
| `momentum_5d` | double | 5-day price change, as a fraction |

**Response — 5 fields per row**

| field | type | meaning |
|-------|------|---------|
| `ticker` | string | echoed back so batch responses are self-describing |
| `prediction` | string | `UP` or `DOWN` for tomorrow's close |
| `probability_up` | double | calibrated probability, 0–1 |
| `confidence` | string | `HIGH` (≥0.65) / `MEDIUM` (≥0.55) / `LOW` |
| `model_tag` | string | which version answered — invaluable during a rollout |

MLflow records this as the model **signature**, and AIDP surfaces it on the deployment's
**Query endpoint** panel. The `input_example` we log becomes the pre-filled sample request there.

In [6]:
from mlflow.models import infer_signature

CONFIDENCE_BANDS = [(0.65, "HIGH"), (0.55, "MEDIUM")]


class StockDirectionModel(mlflow.pyfunc.PythonModel):
    """Frozen serving contract in front of a swappable scikit-learn estimator."""

    def __init__(self, estimator, feature_columns, model_tag):
        self.estimator = estimator
        self.feature_columns = feature_columns
        self.model_tag = model_tag

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        frame = pd.DataFrame(model_input).copy()
        missing = [c for c in self.feature_columns if c not in frame.columns]
        if missing:
            raise ValueError(f"Missing required input columns: {missing}")

        proba = self.estimator.predict_proba(frame[self.feature_columns])[:, 1]
        confidence = [next((label for threshold, label in CONFIDENCE_BANDS
                            if max(p, 1 - p) >= threshold), "LOW") for p in proba]
        proba = np.asarray(proba, dtype="float64").reshape(-1)
        return pd.DataFrame({
            # explicit object-dtype Series built from Python lists: the representation every
            # numpy/pandas/MLflow combination agrees on, on managed runtimes included
            "ticker": pd.Series([str(t) for t in frame["ticker"]], dtype=object),
            "prediction": pd.Series(["UP" if p >= 0.5 else "DOWN" for p in proba], dtype=object),
            "probability_up": np.round(proba, 4).astype("float64"),
            "confidence": pd.Series(confidence, dtype=object),
            "model_tag": pd.Series([str(self.model_tag)] * len(proba), dtype=object),
        })


# Logged with every version: it documents the contract AND pre-fills the AIDP query panel.
INPUT_EXAMPLE = pd.DataFrame([
    {"ticker": "ORCL", "close": 118.42, "sma_5": 116.80, "sma_20": 112.35,
     "rsi_14": 63.4, "volatility_20d": 0.284, "volume_ratio": 1.42, "momentum_5d": 0.0231},
    {"ticker": "TSLA", "close": 214.05, "sma_5": 221.60, "sma_20": 236.90,
     "rsi_14": 34.2, "volatility_20d": 0.551, "volume_ratio": 1.88, "momentum_5d": -0.0418},
])[FEATURE_COLUMNS].astype({"ticker": object})
INPUT_EXAMPLE = with_object_columns(INPUT_EXAMPLE)

print(INPUT_EXAMPLE.to_string(index=False))

ticker  close  sma_5  sma_20  rsi_14  volatility_20d  volume_ratio  momentum_5d
  ORCL 118.42  116.8  112.35    63.4           0.284          1.42       0.0231
  TSLA 214.05  221.6  236.90    34.2           0.551          1.88      -0.0418


## 4 — Create the experiment

One experiment holds every attempt at this problem. In the AIDP UI: **MLOps → Experiments**.

In [7]:
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f"experiment name     : {experiment.name}")
print(f"experiment id       : {experiment.experiment_id}")
print(f"artifact location   : {experiment.artifact_location}")
print(f"lifecycle stage     : {experiment.lifecycle_stage}")
print("\n>>> Show this experiment now under MLOps -> Experiments in the AIDP UI.")

experiment name     : stock_direction_demo_sklearn
experiment id       : 1
artifact location   : mlops-artifact-store:1
lifecycle stage     : active

>>> Show this experiment now under MLOps -> Experiments in the AIDP UI.


## 5 — One training helper, three competing approaches

`train_and_log` is the reusable recipe: fit → evaluate → log params, metrics, tags and dataset
lineage → log the model **with its signature and input example** → register a new version.

`pip_requirements=PIP_REQUIREMENTS` logs the exact `==` pins resolved in section 0 — one explicit,
reproducible `requirements.txt` per version. Unpinned names are the thing that actually breaks a
deployment: the serving container would resolve whatever is latest on the index, and a newer
scikit-learn or numpy than the one that pickled the pipeline fails (or silently changes behaviour)
at load time. Pinning a short list also avoids MLflow freezing every unrelated package in the
workbench image, which is the version of pinning that goes unsatisfiable.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

NUMERIC_INPUTS = [c for c in FEATURE_COLUMNS if c != "ticker"]


def engineer_v2(df):
    """Turn raw price levels into stationary, comparable signals."""
    out = pd.DataFrame(index=df.index)
    out["rsi_dev"] = (df["rsi_14"] - 50) / 15.0
    out["trend_ratio"] = df["sma_5"] / df["sma_20"] - 1.0
    out["trend_dir"] = np.sign(df["sma_5"] / df["sma_20"] - 1.0)
    out["px_vs_sma20"] = df["close"] / df["sma_20"] - 1.0
    out["vol_surprise"] = df["volume_ratio"] - 1.0
    out["volatility_20d"] = df["volatility_20d"]
    out["momentum_5d"] = df["momentum_5d"]
    return out


def engineer_v3(df):
    """v2 plus the explicit interactions that carry most of the edge."""
    out = engineer_v2(df)
    out["heavy_volume"] = (df["volume_ratio"] > 1.5).astype(float)
    out["volume_x_trend"] = out["vol_surprise"] * out["trend_dir"]   # volume confirms or fades
    out["heavy_x_rsi"] = out["heavy_volume"] * out["rsi_dev"]        # exhaustion
    return out


def build_pipeline(estimator, numeric_cols, engineer=None, use_ticker=False):
    numeric_steps = ([("engineer", FunctionTransformer(engineer))] if engineer else []) + [
        ("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
    blocks = [("numeric", Pipeline(numeric_steps), numeric_cols)]
    if use_ticker:
        blocks.append(("ticker", OneHotEncoder(handle_unknown="ignore"), ["ticker"]))
    return Pipeline([("prep", ColumnTransformer(blocks, remainder="drop")), ("clf", estimator)])


def train_and_log(run_name, estimator, numeric_cols, model_tag, notes,
                  engineer=None, use_ticker=False, logged_params=None):
    feature_cols = (["ticker"] if use_ticker else []) + numeric_cols
    pipeline = build_pipeline(estimator, numeric_cols, engineer, use_ticker)

    mlflow.end_run()          # close any run left active by an interrupted cell
    with mlflow.start_run(run_name=run_name) as run:
        pipeline.fit(X_train[feature_cols], y_train)
        proba = pipeline.predict_proba(X_test[feature_cols])[:, 1]
        pred = (proba >= 0.5).astype(int)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "precision": float(precision_score(y_test, pred, zero_division=0)),
            "recall": float(recall_score(y_test, pred, zero_division=0)),
            "f1": float(f1_score(y_test, pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_test, proba)),
            "lift_over_baseline_pct": float((accuracy_score(y_test, pred) - BASELINE_ACCURACY) * 100),
        }

        mlflow.log_params({
            "algorithm": type(estimator).__name__,
            "feature_count": len(feature_cols),
            "uses_ticker_identity": use_ticker,
            "feature_engineering": "none" if engineer is None else engineer.__name__,
            "train_rows": len(X_train), "test_rows": len(X_test),
            "train_cutoff": f"{split_date:%Y-%m-%d}",
            **(logged_params or {}),
        })
        mlflow.log_metrics(metrics)
        mlflow.set_tags({"demo": "aidp-mlops-stock-sklearn", "model_tag": model_tag,
                         "approach": notes, "universe": ",".join(TICKERS)})

        # Dataset lineage: the run records what it was trained and evaluated on.
        mlflow.log_input(mlflow.data.from_pandas(train_df, name="stock_features_train",
                                                 targets="target_up"), context="training")
        mlflow.log_input(mlflow.data.from_pandas(test_df, name="stock_features_test",
                                                 targets="target_up"), context="evaluation")

        served = StockDirectionModel(pipeline, feature_cols, model_tag)
        signature = infer_signature(INPUT_EXAMPLE, served.predict(None, INPUT_EXAMPLE))

        model_info = mlflow.pyfunc.log_model(
            name="stock_direction_model",
            python_model=served,
            signature=signature,               # <- the input/output contract
            input_example=INPUT_EXAMPLE,       # <- pre-fills the AIDP query panel
            registered_model_name=MODEL_NAME,  # <- creates the next registry version
            pip_requirements=PIP_REQUIREMENTS,   # <- exact "==" pins from section 0
        )

        print(f"[{run_name}]  accuracy={metrics['accuracy']:.3f}  roc_auc={metrics['roc_auc']:.3f}"
              f"  f1={metrics['f1']:.3f}  (+{metrics['lift_over_baseline_pct']:.1f} pts vs baseline)")
        print(f"   run_id    : {run.info.run_id}")
        print(f"   model_uri : {model_info.model_uri}")
        return {"run_id": run.info.run_id, "model_info": model_info,
                "metrics": metrics, "model_tag": model_tag}

### Run 1 — the honest baseline

Logistic regression on raw prices and RSI. No volume, no ticker identity. This is what a first
afternoon of work looks like.

In [9]:
run_v1 = train_and_log(
    run_name="v1-logistic-baseline",
    estimator=LogisticRegression(C=1.0, max_iter=1000),
    numeric_cols=["close", "rsi_14"],
    model_tag="v1-logreg",
    notes="Baseline: raw close + RSI, linear, ticker-agnostic",
    logged_params={"hp_C": 1.0, "hp_max_iter": 1000},
)

[v1-logistic-baseline]  accuracy=0.602  roc_auc=0.641  f1=0.620  (+9.8 pts vs baseline)
   run_id    : 63e7bbdca5be4039be452ae1574ad315
   model_uri : models:/m-1a1c207c007c471b952e5eef8c526ebe
🏃 View run v1-logistic-baseline at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1/runs/63e7bbdca5be4039be452ae1574ad315
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1


### Run 2 — add volume, volatility and ticker identity

A random forest over stationary engineered features, plus a one-hot of the ticker so the model can
learn that NVDA and ORCL behave differently on heavy volume.

In [10]:
run_v2 = train_and_log(
    run_name="v2-random-forest",
    estimator=RandomForestClassifier(n_estimators=400, max_depth=10,
                                     min_samples_leaf=20, random_state=42, n_jobs=-1),
    numeric_cols=NUMERIC_INPUTS,
    model_tag="v2-rf",
    notes="Full feature set, stationary transforms, ticker one-hot",
    engineer=engineer_v2,
    use_ticker=True,
    logged_params={"hp_n_estimators": 400, "hp_max_depth": 10, "hp_min_samples_leaf": 20},
)

[v2-random-forest]  accuracy=0.620  roc_auc=0.656  f1=0.628  (+11.6 pts vs baseline)
   run_id    : 2ca33d4703a544b9913e71b7a5dbea56
   model_uri : models:/m-b8801806aa7645cc8a9af0785fa4773c


🏃 View run v2-random-forest at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1/runs/2ca33d4703a544b9913e71b7a5dbea56
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1


### Run 3 — tuned gradient boosting with explicit interactions

Same contract, same inputs — but the interactions that actually drive the edge are handed to the
model directly, and the estimator is a tuned histogram gradient booster with early stopping.

In [11]:
run_v3 = train_and_log(
    run_name="v3-gradient-boosting",
    estimator=HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=400, max_leaf_nodes=31, min_samples_leaf=30,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, random_state=42),
    numeric_cols=NUMERIC_INPUTS,
    model_tag="v3-hgb",
    notes="Tuned gradient boosting + explicit volume/trend interactions",
    engineer=engineer_v3,
    use_ticker=True,
    logged_params={"hp_learning_rate": 0.05, "hp_max_iter": 400,
                   "hp_max_leaf_nodes": 31, "hp_l2_regularization": 1.0},
)

[v3-gradient-boosting]  accuracy=0.623  roc_auc=0.663  f1=0.632  (+11.9 pts vs baseline)
   run_id    : 2889351725404d9d8622ef1a88098aa1
   model_uri : models:/m-9fe4661391fb42aeb63d446bcb79a285
🏃 View run v3-gradient-boosting at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1/runs/2889351725404d9d8622ef1a88098aa1
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/1


## 6 — The leaderboard

`mlflow.search_runs` is the same query the **Experiments** grid runs in the UI. Sort by ROC-AUC and
the winner is obvious.

In [12]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], order_by=["metrics.roc_auc DESC"])
leaderboard = (runs[["tags.mlflow.runName", "params.algorithm", "params.feature_count",
                     "params.uses_ticker_identity", "metrics.accuracy", "metrics.roc_auc",
                     "metrics.f1", "metrics.lift_over_baseline_pct"]]
               .rename(columns=lambda c: c.split(".")[-1])
               .round(4))
print(leaderboard.to_string(index=False))

versions = sorted(client.search_model_versions(f"name = '{MODEL_NAME}'"),
                  key=lambda v: int(v.version))
run_to_version = {v.run_id: v.version for v in versions}
for label, r in [("v1", run_v1), ("v2", run_v2), ("v3", run_v3)]:
    r["version"] = run_to_version.get(r["run_id"])
print("\nregistry versions:",
      {label: r["version"] for label, r in [("v1", run_v1), ("v2", run_v2), ("v3", run_v3)]})

             runName                      algorithm feature_count uses_ticker_identity  accuracy  roc_auc     f1  lift_over_baseline_pct
v3-gradient-boosting HistGradientBoostingClassifier             8                 True    0.6229   0.6632 0.6323                 11.8708
    v2-random-forest         RandomForestClassifier             8                 True    0.6204   0.6557 0.6283                 11.6191
v1-logistic-baseline             LogisticRegression             2                False    0.6023   0.6414 0.6202                  9.8154



registry versions: {"v1": "1", "v2": "2", "v3": "3"}


### Promote a champion and a challenger

Descriptions and tags travel with the version; **aliases** (`champion`, `challenger`) let clients
reference a role instead of a number, so a promotion never requires a client change.

In [13]:
ranked = sorted([run_v1, run_v2, run_v3], key=lambda r: r["metrics"]["roc_auc"], reverse=True)
champion, challenger = ranked[0], ranked[1]

for r, role in [(champion, "champion"), (challenger, "challenger"), (ranked[2], "archived")]:
    client.update_model_version(
        name=MODEL_NAME, version=r["version"],
        description=(f"{r['model_tag']} | accuracy={r['metrics']['accuracy']:.3f} "
                     f"roc_auc={r['metrics']['roc_auc']:.3f} | role={role}"))
    client.set_model_version_tag(MODEL_NAME, r["version"], "role", role)
    client.set_model_version_tag(MODEL_NAME, r["version"], "model_tag", r["model_tag"])

for r, alias in [(champion, "champion"), (challenger, "challenger")]:
    try:
        client.set_registered_model_alias(MODEL_NAME, alias, r["version"])
        print(f"alias '{alias}' -> version {r['version']} ({r['model_tag']})")
    except Exception as exc:
        print(f"alias '{alias}' not set ({type(exc).__name__}); the 'role' tag covers the demo")

print(f"\nCHAMPION   : version {champion['version']} ({champion['model_tag']}) "
      f"roc_auc={champion['metrics']['roc_auc']:.3f}")
print(f"CHALLENGER : version {challenger['version']} ({challenger['model_tag']}) "
      f"roc_auc={challenger['metrics']['roc_auc']:.3f}")
print("\n>>> Show MLOps -> Models -> " + MODEL_NAME + " : three versions, one lineage.")

alias "champion" -> version 3 (v3-hgb)


alias "challenger" -> version 2 (v2-rf)

CHAMPION   : version 3 (v3-hgb) roc_auc=0.663
CHALLENGER : version 2 (v2-rf) roc_auc=0.656

>>> Show MLOps -> Models -> default.default.stock_direction_predictor_sklearn : three versions, one lineage.


## 7 — The contract, exactly as the platform reads it

This is what AIDP stores as `input_contract` / `output_contract` when a deployment is activated,
and what the **Query endpoint** panel renders. The same cell reads back each version's logged
`requirements.txt` — the environment spec the serving container installs from — so you can confirm
every dependency is pinned before anyone clicks *Activate*.

In [14]:
for r in [run_v1, run_v2, run_v3]:
    info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{r['version']}")
    sig = info.signature.to_dict()
    print(f"--- version {r['version']}  ({r['model_tag']}) ---")
    print("  inputs :", json.dumps(json.loads(sig["inputs"]), separators=(",", ":")))
    print("  outputs:", json.dumps(json.loads(sig["outputs"]), separators=(",", ":")))
    print()

print("Contracts identical across all three versions:",
      len({mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{r['version']}").signature.to_dict()["inputs"]
           for r in [run_v1, run_v2, run_v3]}) == 1)

# The logged environment, straight from the model artifacts the serving container reads.
for r in [run_v1, run_v2, run_v3]:
    logged = [line.strip() for line in mlflow.artifacts.load_text(
        f"models:/{MODEL_NAME}/{r['version']}/requirements.txt").splitlines() if line.strip()]
    unpinned = [line for line in logged if not line.startswith("#") and "==" not in line]
    print(f"--- version {r['version']} requirements.txt ---")
    print("  " + "\n  ".join(logged))
    print(f"  fully pinned: {not unpinned}" + (f"  (unpinned: {unpinned})" if unpinned else ""))

--- version 1  (v1-logreg) ---
  inputs : [{"type":"string","name":"ticker","required":true},{"type":"double","name":"close","required":true},{"type":"double","name":"sma_5","required":true},{"type":"double","name":"sma_20","required":true},{"type":"double","name":"rsi_14","required":true},{"type":"double","name":"volatility_20d","required":true},{"type":"double","name":"volume_ratio","required":true},{"type":"double","name":"momentum_5d","required":true}]
  outputs: [{"type":"string","name":"ticker","required":true},{"type":"string","name":"prediction","required":true},{"type":"double","name":"probability_up","required":true},{"type":"string","name":"confidence","required":true},{"type":"string","name":"model_tag","required":true}]



--- version 2  (v2-rf) ---
  inputs : [{"type":"string","name":"ticker","required":true},{"type":"double","name":"close","required":true},{"type":"double","name":"sma_5","required":true},{"type":"double","name":"sma_20","required":true},{"type":"double","name":"rsi_14","required":true},{"type":"double","name":"volatility_20d","required":true},{"type":"double","name":"volume_ratio","required":true},{"type":"double","name":"momentum_5d","required":true}]
  outputs: [{"type":"string","name":"ticker","required":true},{"type":"string","name":"prediction","required":true},{"type":"double","name":"probability_up","required":true},{"type":"string","name":"confidence","required":true},{"type":"string","name":"model_tag","required":true}]



--- version 3  (v3-hgb) ---
  inputs : [{"type":"string","name":"ticker","required":true},{"type":"double","name":"close","required":true},{"type":"double","name":"sma_5","required":true},{"type":"double","name":"sma_20","required":true},{"type":"double","name":"rsi_14","required":true},{"type":"double","name":"volatility_20d","required":true},{"type":"double","name":"volume_ratio","required":true},{"type":"double","name":"momentum_5d","required":true}]
  outputs: [{"type":"string","name":"ticker","required":true},{"type":"string","name":"prediction","required":true},{"type":"double","name":"probability_up","required":true},{"type":"string","name":"confidence","required":true},{"type":"string","name":"model_tag","required":true}]



Contracts identical across all three versions: True


--- version 1 requirements.txt ---
  mlflow==3.12.0
  scikit-learn==1.8.0
  numpy==2.4.6
  scipy==1.17.1
  pandas==2.2.3
  cloudpickle==3.1.2
  fully pinned: True


--- version 2 requirements.txt ---
  mlflow==3.12.0
  scikit-learn==1.8.0
  numpy==2.4.6
  scipy==1.17.1
  pandas==2.2.3
  cloudpickle==3.1.2
  fully pinned: True


--- version 3 requirements.txt ---
  mlflow==3.12.0
  scikit-learn==1.8.0
  numpy==2.4.6
  scipy==1.17.1
  pandas==2.2.3
  cloudpickle==3.1.2
  fully pinned: True


## 8 — Score all three versions on the same morning batch

Load each registered version straight from the registry and ask the identical question. Same
request in, same response shape out — only `model_tag` and the numbers change.

In [15]:
MORNING_BATCH = pd.DataFrame([
    # ticker,  close,   sma_5,   sma_20,  rsi_14, volatility_20d, volume_ratio, momentum_5d
    ("ORCL",  118.42,  116.80,  112.35,   63.4,   0.284,          1.42,         0.0231),
    ("AAPL",  228.10,  231.45,  236.80,   38.7,   0.242,          0.86,        -0.0188),
    ("NVDA",  892.55,  861.20,  810.40,   71.2,   0.478,          1.94,         0.0512),
    ("TSLA",  214.05,  221.60,  236.90,   34.2,   0.551,          1.88,        -0.0418),
    ("JPM",   198.30,  197.85,  195.10,   54.9,   0.203,          0.97,         0.0064),
    ("MSFT",  432.70,  428.15,  419.60,   58.3,   0.238,          1.11,         0.0155),
], columns=FEATURE_COLUMNS)

loaded = {}
comparison = MORNING_BATCH[["ticker", "close", "rsi_14", "volume_ratio"]].copy()
for r in [run_v1, run_v2, run_v3]:
    model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{r['version']}")
    loaded[r["model_tag"]] = model
    out = model.predict(MORNING_BATCH)
    comparison[f"v{r['version']}_call"] = out["prediction"].tolist()
    comparison[f"v{r['version']}_p_up"] = out["probability_up"].to_numpy(dtype="float64")

print(comparison.to_string(index=False))

disagree = comparison[comparison.filter(like="_call").nunique(axis=1) > 1]["ticker"].tolist()
print(f"\nVersions disagree on: {disagree or 'nothing in this batch'}"
      "  <- exactly why you keep every version and A/B them.")

ticker  close  rsi_14  volume_ratio v1_call  v1_p_up v2_call  v2_p_up v3_call  v3_p_up
  ORCL 118.42    63.4          1.42    DOWN   0.3568    DOWN   0.2749    DOWN   0.2651
  AAPL 228.10    38.7          0.86      UP   0.6684      UP   0.6338      UP   0.6081
  NVDA 892.55    71.2          1.94    DOWN   0.2609      UP   0.5627      UP   0.6177
  TSLA 214.05    34.2          1.88      UP   0.7187    DOWN   0.4037    DOWN   0.2051
   JPM 198.30    54.9          0.97    DOWN   0.4632    DOWN   0.4591    DOWN   0.3952
  MSFT 432.70    58.3          1.11    DOWN   0.4161    DOWN   0.3563    DOWN   0.3490

Versions disagree on: ["NVDA", "TSLA"]  <- exactly why you keep every version and A/B them.


## 9 — Deploy the champion, then call it

**In the AIDP UI:** `MLOps → Models → default.default.stock_direction_predictor_sklearn →`
**Deployments → Create deployment** — pick the champion version, a compute, then **Activate**.
Once it is `ACTIVE` the deployment exposes a serving URI and a **Query endpoint** panel
pre-filled from the input example logged above.

**The wire contract**

* Request body — MLflow *split* orient: `{"columns": [...], "data": [[...], ...]}`
  (a plain list of row objects also works)
* Response body — `{"predictions": [ {…}, … ]}`, one object per input row

MLflow writes the logged input example to `input_example.json` in exactly that split shape, and that
is the file AIDP reads for the Query endpoint panel — so the pre-filled sample request is directly
sendable, with no reshaping.

The cell below builds three demo requests and produces the **exact responses** the deployed endpoint
will return, by running the champion through the same serialisation the AIDP serving runtime uses.

In [16]:
def to_serving_request(frame):
    """MLflow split orient - the request body the AIDP model endpoint expects."""
    return {"columns": list(frame.columns),
            "data": [[None if pd.isna(v) else v for v in row] for row in frame.to_numpy().tolist()]}


def to_serving_response(predictions):
    """The {"predictions": [...]} envelope the AIDP serving runtime returns."""
    records = predictions.to_dict(orient="records")
    clean = [{k: (None if (isinstance(v, float) and not np.isfinite(v)) else
                  v.item() if hasattr(v, "item") else v) for k, v in row.items()}
             for row in records]
    return {"predictions": clean}


champion_model = loaded[champion["model_tag"]]

SCENARIOS = {
    "1. AAPL - oversold (RSI 38.7) on quiet volume": MORNING_BATCH[MORNING_BATCH.ticker == "AAPL"],
    "2. TSLA - just as oversold (RSI 34.2), but heavy volume in a downtrend":
        MORNING_BATCH[MORNING_BATCH.ticker == "TSLA"],
    "3. Full morning batch - six names in a single call": MORNING_BATCH,
}

demo_payloads = {}
for title, frame in SCENARIOS.items():
    request = to_serving_request(frame.reset_index(drop=True))
    response = to_serving_response(champion_model.predict(frame))
    demo_payloads[title] = {"request": request, "response": response}

    print("=" * 100)
    print(title)
    print("=" * 100)
    print("REQUEST  POST {serving_uri}/predict")
    print(json.dumps(request, indent=2))
    print("\nRESPONSE  200 OK")
    print(json.dumps(response, indent=2))
    print()

a, b = list(demo_payloads.values())[0], list(demo_payloads.values())[1]
print("THE PUNCHLINE ->  two equally oversold names, opposite calls:")
print(f"   AAPL  {a['response']['predictions'][0]['prediction']:<4} "
      f"p_up={a['response']['predictions'][0]['probability_up']:.3f}  (quiet volume: the dip gets bought)")
print(f"   TSLA  {b['response']['predictions'][0]['prediction']:<4} "
      f"p_up={b['response']['predictions'][0]['probability_up']:.3f}  (heavy volume: the selloff has legs)")
print("   Same contract, same request shape - the model learned that volume changes the answer.\n")

with open("stock_model_demo_payloads.json", "w") as fh:
    json.dump({"model": MODEL_NAME, "champion_version": champion["version"],
               "model_tag": champion["model_tag"], "scenarios": demo_payloads}, fh, indent=2)
client.log_artifact(champion["run_id"], "stock_model_demo_payloads.json")
print("Saved + logged to the champion run: stock_model_demo_payloads.json")
print("(paste any 'request' straight into the deployment's Query endpoint panel)")

1. AAPL - oversold (RSI 38.7) on quiet volume
REQUEST  POST {serving_uri}/predict
{
  "columns": [
    "ticker",
    "close",
    "sma_5",
    "sma_20",
    "rsi_14",
    "volatility_20d",
    "volume_ratio",
    "momentum_5d"
  ],
  "data": [
    [
      "AAPL",
      228.1,
      231.45,
      236.8,
      38.7,
      0.242,
      0.86,
      -0.0188
    ]
  ]
}

RESPONSE  200 OK
{
  "predictions": [
    {
      "ticker": "AAPL",
      "prediction": "UP",
      "probability_up": 0.6081,
      "confidence": "MEDIUM",
      "model_tag": "v3-hgb"
    }
  ]
}

2. TSLA - just as oversold (RSI 34.2), but heavy volume in a downtrend
REQUEST  POST {serving_uri}/predict
{
  "columns": [
    "ticker",
    "close",
    "sma_5",
    "sma_20",
    "rsi_14",
    "volatility_20d",
    "volume_ratio",
    "momentum_5d"
  ],
  "data": [
    [
      "TSLA",
      214.05,
      221.6,
      236.9,
      34.2,
      0.551,
      1.88,
      -0.0418
    ]
  ]
}

RESPONSE  200 OK
{
  "predictions": [
    

Saved + logged to the champion run: stock_model_demo_payloads.json
(paste any "request" straight into the deployment"s Query endpoint panel)


### The same call from outside the notebook

Copy the serving URI from the deployment page. Under `AIDP_WORKBENCH` auth the endpoint takes your
workbench bearer token; under `OAUTH` it takes the configured client credentials.

In [17]:
single = demo_payloads["1. AAPL - oversold (RSI 38.7) on quiet volume"]["request"]

print("# --- curl ---------------------------------------------------------------")
print(f"""curl -X POST "$SERVING_URI/predict" \\
  -H "Authorization: Bearer $AIDP_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{json.dumps(single)}'""")

print("\n# --- python -------------------------------------------------------------")
snippet = f"""import os, requests

response = requests.post(
    f"{{os.environ['SERVING_URI']}}/predict",
    headers={{"Authorization": f"Bearer {{os.environ['AIDP_TOKEN']}}",
             "Content-Type": "application/json"}},
    json={json.dumps(single)},
    timeout=30,
)
print(response.status_code, response.json())"""
print(snippet)
print("# -> 200", json.dumps(demo_payloads["1. AAPL - oversold (RSI 38.7) on quiet volume"]["response"]))

# --- curl ---------------------------------------------------------------
curl -X POST "$SERVING_URI/predict" \
  -H "Authorization: Bearer $AIDP_TOKEN" \
  -H "Content-Type: application/json" \
  -d "{"columns": ["ticker", "close", "sma_5", "sma_20", "rsi_14", "volatility_20d", "volume_ratio", "momentum_5d"], "data": [["AAPL", 228.1, 231.45, 236.8, 38.7, 0.242, 0.86, -0.0188]]}"

# --- python -------------------------------------------------------------
import os, requests

response = requests.post(
    f"{os.environ["SERVING_URI"]}/predict",
    headers={"Authorization": f"Bearer {os.environ["AIDP_TOKEN"]}",
             "Content-Type": "application/json"},
    json={"columns": ["ticker", "close", "sma_5", "sma_20", "rsi_14", "volatility_20d", "volume_ratio", "momentum_5d"], "data": [["AAPL", 228.1, 231.45, 236.8, 38.7, 0.242, 0.86, -0.0188]]},
    timeout=30,
)
print(response.status_code, response.json())
# -> 200 {"predictions": [{"ticker": "AAPL", "prediction": "UP", "probability

## Demo talk track — eight minutes, in order

1. **Set the scene** *(30s)* — "Eight household names. One question: does it close higher tomorrow?"
2. **Data & features** *(1m)* — run cells 1–2, point at the leaderboard-ready base rate: ~50%.
   Anything above that is real edge.
3. **Experiments** *(2m)* — run the three training cells, then open **MLOps → Experiments** in the
   UI. Show params, metrics, tags and dataset lineage captured with no extra plumbing.
4. **Registry** *(1m)* — **MLOps → Models**: one governed `catalog.schema.name`, three versions,
   each traceable back to its run, with champion / challenger roles.
5. **The contract** *(1m)* — show the identical input/output signature across all three versions.
   *"The algorithm changed three times. The API never did."*
6. **Compare** *(1m)* — the side-by-side prediction table, and the tickers where the versions
   disagree.
7. **Deploy & query** *(2m)* — create the deployment on the champion, activate it, open the
   **Query endpoint** panel (already pre-filled from `input_example`), send the ORCL request, read
   the response, then swap in the six-name batch.
8. **Land it** *(30s)* — *"Train, track, register, govern, deploy and serve — one platform, one
   notebook, no glue code."*

**Two questions worth planting in the room**

* *"What if the challenger turns out to be better in production?"* → roll forward the deployment to
  another version; `model_tag` in every response tells you which version served it.
* *"How do we know what this endpoint accepts?"* → the contract is derived from the model itself,
  not from a wiki page that drifts.

## Optional — clean up

Leave `CLEANUP = False` during the demo. Flip it to `True` afterwards to remove the registered
model and experiment created by this notebook. Delete any **deployment** from the UI first — an
active deployment holds a reference to its model version.

In [18]:
CLEANUP = False

if CLEANUP:
    for version in client.search_model_versions(f"name = '{MODEL_NAME}'"):
        try:
            client.delete_model_version(MODEL_NAME, version.version)
            print("deleted model version", version.version)
        except Exception as exc:
            print("skip version", version.version, type(exc).__name__, exc)
    try:
        client.delete_registered_model(MODEL_NAME)
        print("deleted registered model", MODEL_NAME)
    except Exception as exc:
        print("skip registered model:", type(exc).__name__, exc)
    try:
        client.delete_experiment(mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id)
        print("deleted experiment", EXPERIMENT_NAME)
    except Exception as exc:
        print("skip experiment:", type(exc).__name__, exc)
else:
    print("CLEANUP is False - nothing removed.")

CLEANUP is False - nothing removed.
